[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/02_root_finding_methods/first_principles.ipynb)

# Topic 02: Root-Finding Methods

## 1. First-Principles Intuition & Motivation

An enormous fraction of applied mathematics reduces to one deceptively simple task: given a continuous function $f$, find $x^{*}$ with

$$
f(x^{*}) = 0 .
$$

Equilibria of dynamical systems, break-even points, implied volatilities, orbital anomalies, optimality conditions $\nabla f = 0$, eigenvalues (roots of characteristic polynomials) — all are root-finding problems. Beyond degree four there is no general closed-form solution even for polynomials (Abel–Ruffini), so *iteration* is not a convenience but a necessity.

Every root-finding method is built from the same two-step logic:

1. **Replace** $f$ locally by a model simple enough to solve exactly (a sign bracket, a tangent line, a secant line).
2. **Iterate**: solve the model, move to the solution, rebuild the model.

The methods differ only in the model, and the model dictates everything: how fast the iteration converges, what information it needs ($f$ values? derivatives?), and how it can fail.

### The central trade-off: robustness vs speed

- **Bisection** uses the weakest possible model — a sign change — and in exchange gets an unconditional guarantee: it *cannot* fail on a valid bracket. Price: one bit of accuracy per function evaluation.
- **Newton's method** uses the strongest local model — the tangent line — and converges *quadratically*, doubling the number of correct digits per step. Price: it needs $f'$, and far from the root it can diverge, cycle, or shoot off to infinity.
- **Secant** replaces the tangent by a chord through the last two iterates: derivative-free, superlinear order $\varphi = (1+\sqrt{5})/2 \approx 1.618$ — and per *function evaluation* often faster than Newton.

Production solvers (Brent's method) hybridize: maintain a bisection bracket for safety, attempt secant/inverse-quadratic steps for speed, and fall back when a fast step leaves the bracket.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Root, simple root, multiplicity).** $x^{*}$ is a *root* of $f$ if $f(x^{*}) = 0$. If $f$ is $m$ times continuously differentiable with

$$
f(x^{*}) = f'(x^{*}) = \cdots = f^{(m-1)}(x^{*}) = 0, \qquad f^{(m)}(x^{*}) \neq 0,
$$

then $x^{*}$ has *multiplicity* $m$; a root of multiplicity 1 is *simple* ($f'(x^{*}) \neq 0$).

**Definition 2 (Order of convergence).** A sequence $x_n \to x^{*}$ with errors $e_n = \lvert x_n - x^{*} \rvert$ converges with *order* $p \ge 1$ and *asymptotic constant* $C$ if

$$
\lim_{n \to \infty} \frac{e_{n+1}}{e_n^{\,p}} = C, \qquad 0 \lt C \lt \infty
$$

(with $C \lt 1$ required when $p = 1$, called *linear* convergence; $1 \lt p \lt 2$ is *superlinear*, $p = 2$ *quadratic*).

**Theorem 1 (Bolzano / Intermediate Value Theorem).** If $f \in C[a, b]$ and $f(a)f(b) \lt 0$, then there exists $x^{*} \in (a, b)$ with $f(x^{*}) = 0$.

**Theorem 2 (Bisection convergence).** With $[a_0, b_0]$ a valid bracket and $c_n$ the midpoints, there is a root $x^{*}$ with

$$
\lvert c_n - x^{*} \rvert \le \frac{b_0 - a_0}{2^{\,n+1}} .
$$

**Theorem 3 (Newton's local quadratic convergence).** Let $f \in C^{2}$ near a simple root $x^{*}$. Then there is $\delta \gt 0$ such that for every $x_0$ with $\lvert x_0 - x^{*} \rvert \le \delta$, the iteration

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}
$$

converges to $x^{*}$ and

$$
\lim_{n \to \infty} \frac{\lvert x_{n+1} - x^{*} \rvert}{\lvert x_n - x^{*} \rvert^{2}} = \left\lvert \frac{f''(x^{*})}{2 f'(x^{*})} \right\rvert .
$$

**Theorem 4 (Secant convergence).** Under the hypotheses of Theorem 3, the secant iteration

$$
x_{n+1} = x_n - f(x_n)\,\frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}
$$

converges locally with order $\varphi = \frac{1 + \sqrt{5}}{2} \approx 1.618$, and the errors satisfy $e_{n+1} \approx \left\lvert \frac{f''(x^{*})}{2f'(x^{*})} \right\rvert e_n e_{n-1}$.

**Theorem 5 (Newton at a multiple root).** If $x^{*}$ has multiplicity $m \ge 2$, Newton's method converges only *linearly* with factor $\frac{m-1}{m}$; the modified iteration $x_{n+1} = x_n - m\,\frac{f(x_n)}{f'(x_n)}$ restores quadratic convergence.

**Theorem 6 (Global Newton for convex functions).** If $f \in C^{2}[x^{*}, \infty)$ is increasing and convex with $f(x^{*}) = 0$, then Newton's method converges to $x^{*}$ monotonically from *any* starting point $x_0 \gt x^{*}$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — Bisection error bound and iteration count

**Claim.** $\lvert c_n - x^{*} \rvert \le \dfrac{b_0 - a_0}{2^{n+1}}$, and reaching tolerance $\varepsilon$ requires at most $n \ge \log_2\dfrac{b_0 - a_0}{\varepsilon} - 1$ iterations.

**Proof.** By construction each step keeps the sign change: $f(a_n)f(b_n) \le 0$, so by Theorem 1 a root $x^{*}$ lies in every $[a_n, b_n]$. The interval halves exactly:

$$
b_n - a_n = \frac{b_0 - a_0}{2^{n}} .
$$

The midpoint $c_n = \frac{a_n + b_n}{2}$ is within half the interval of any point of it, in particular of $x^{*}$:

$$
\lvert c_n - x^{*} \rvert \le \frac{b_n - a_n}{2} = \frac{b_0 - a_0}{2^{n+1}} .
$$

Requiring the bound to be $\le \varepsilon$ and solving for $n$ gives $2^{n+1} \ge (b_0 - a_0)/\varepsilon$, i.e. $n \ge \log_2\frac{b_0 - a_0}{\varepsilon} - 1$. Convergence is linear with rate $\tfrac{1}{2}$: one binary digit per iteration, unconditionally. $\blacksquare$

### Proof 2 — Newton's method: quadratic convergence

**Claim.** (Theorem 3.) For a simple root and $f \in C^2$, Newton's errors satisfy $e_{n+1} = \left\lvert \frac{f''(\xi_n)}{2f'(x_n)} \right\rvert e_n^2$ and the method converges for close enough starts.

**Proof.** Taylor-expand $f$ about the current iterate $x_n$, evaluated at the root, with Lagrange remainder: for some $\xi_n$ between $x_n$ and $x^{*}$,

$$
0 = f(x^{*}) = f(x_n) + f'(x_n)(x^{*} - x_n) + \tfrac{1}{2} f''(\xi_n)(x^{*} - x_n)^2 .
$$

Divide by $f'(x_n)$ (nonzero near the simple root by continuity) and rearrange:

$$
\underbrace{x_n - \frac{f(x_n)}{f'(x_n)}}_{x_{n+1}} - \,x^{*} = \frac{f''(\xi_n)}{2 f'(x_n)} (x_n - x^{*})^2 .
$$

Taking absolute values, $e_{n+1} = \left\lvert \frac{f''(\xi_n)}{2f'(x_n)} \right\rvert e_n^2$. Choose $\delta$ small enough that $M := \sup_{\lvert x - x^{*} \rvert \le \delta} \left\lvert \frac{f''(x)}{2f'(x)} \right\rvert$ satisfies $M\delta \le \tfrac{1}{2}$. Then $e_0 \le \delta$ implies $e_1 \le M e_0^2 \le \tfrac{1}{2} e_0 \le \delta$: the iterates stay in the ball and $e_n \le \frac{1}{M} (M e_0)^{2^{n}} \to 0$ doubly exponentially. Letting $n \to \infty$, $\xi_n \to x^{*}$ gives the asymptotic constant $\left\lvert \frac{f''(x^{*})}{2f'(x^{*})} \right\rvert$. $\blacksquare$

The bound $e_n \le \frac{1}{M}(Me_0)^{2^n}$ is the precise meaning of "digits double each step."

### Proof 3 — Secant method: the golden-ratio order

**Claim.** Secant errors obey $e_{n+1} \approx C\, e_n e_{n-1}$ with $C = \left\lvert \frac{f''(x^{*})}{2f'(x^{*})} \right\rvert$, which yields convergence order $\varphi = \frac{1+\sqrt{5}}{2}$.

**Derivation.** The secant step is Newton with $f'(x_n)$ replaced by the divided difference $f[x_{n-1}, x_n]$. Using the error form of linear interpolation (Topic 04): writing the secant line through $(x_{n-1}, f(x_{n-1}))$ and $(x_n, f(x_n))$ and evaluating the interpolation error at $x^{*}$, one obtains after algebra

$$
x_{n+1} - x^{*} = (x_n - x^{*})(x_{n-1} - x^{*})\, \frac{f[x_{n-1}, x_n, x^{*}]}{f[x_{n-1}, x_n]} \approx \frac{f''(x^{*})}{2 f'(x^{*})}\,(x_n - x^{*})(x_{n-1} - x^{*}),
$$

since the second-order divided difference tends to $\tfrac{1}{2}f''(x^{*})$ and the first-order one to $f'(x^{*})$. Hence $e_{n+1} \approx C e_n e_{n-1}$.

**Order extraction.** Posit $e_{n+1} \approx K e_n^{\,p}$. Substituting into $e_{n+1} = C e_n e_{n-1}$ and using $e_n = K e_{n-1}^{\,p}$:

$$
K e_n^{\,p} = C e_n e_{n-1} = C e_n (e_n / K)^{1/p} \implies e_n^{\,p} \propto e_n^{\,1 + 1/p} \implies p = 1 + \frac{1}{p} \implies p^2 - p - 1 = 0 .
$$

The positive solution is $p = \frac{1 + \sqrt{5}}{2} = \varphi$, with $K = C^{1/\varphi}$ fixed by consistency. $\blacksquare$

### Proof 4 — Newton at a multiple root converges only linearly

**Claim.** (Theorem 5.) If $x^{*}$ has multiplicity $m \ge 2$, then $e_{n+1} \approx \frac{m-1}{m} e_n$.

**Proof.** Near $x^{*}$ write $f(x) = (x - x^{*})^{m} g(x)$ with $g(x^{*}) = \frac{f^{(m)}(x^{*})}{m!} \neq 0$ and $g$ continuous. Then

$$
f'(x) = m (x - x^{*})^{m-1} g(x) + (x - x^{*})^{m} g'(x),
$$

so the Newton correction is

$$
\frac{f(x)}{f'(x)} = \frac{(x - x^{*})\, g(x)}{m\, g(x) + (x - x^{*}) g'(x)} .
$$

With $e_n = x_n - x^{*}$ (signed), the iteration gives

$$
e_{n+1} = e_n - \frac{e_n g(x_n)}{m g(x_n) + e_n g'(x_n)} = e_n \left( 1 - \frac{1}{m} + O(e_n) \right).
$$

Thus $e_{n+1}/e_n \to \frac{m-1}{m} \lt 1$: linear convergence, painfully slow for large $m$ (e.g. factor $\tfrac{1}{2}$ for a double root — no better than bisection). Replacing the correction by $m f/f'$ cancels the factor $\bigl(1 - \tfrac{1}{m}\bigr)$ exactly, and the next-order term gives quadratic convergence. $\blacksquare$

### Proof 5 — Global convergence of Newton for increasing convex functions

**Claim.** (Theorem 6.) If $f' \gt 0$ and $f'' \ge 0$ on $[x^{*}, \infty)$ and $f(x^{*}) = 0$, then from any $x_0 \gt x^{*}$ Newton's iterates decrease monotonically to $x^{*}$.

**Proof.** *Step 1: iterates stay right of the root.* For $x_n \gt x^{*}$, convexity puts the graph above every tangent line; evaluating the tangent at $x^{*}$:

$$
0 = f(x^{*}) \ge f(x_n) + f'(x_n)(x^{*} - x_n) \implies x^{*} \le x_n - \frac{f(x_n)}{f'(x_n)} = x_{n+1} .
$$

*Step 2: iterates decrease.* Since $f$ is increasing and $x_n \gt x^{*}$, $f(x_n) \gt 0$, so the correction $f(x_n)/f'(x_n) \gt 0$ and $x_{n+1} \lt x_n$.

*Step 3: convergence.* The sequence $(x_n)$ is decreasing and bounded below by $x^{*}$, hence converges to some $L \ge x^{*}$. Passing to the limit in the iteration (all functions continuous, $f'(L) \gt 0$) gives $L = L - f(L)/f'(L)$, so $f(L) = 0$ and by strict monotonicity $L = x^{*}$. $\blacksquare$

This is why Newton on $x^2 - a$ (square roots) and on Kepler's equation (with the right bracket) is globally reliable.

## 4. Computational & Algorithmic Insights

### Stopping criteria — the subtle part

Three tests are in common use, and none is universally right:

$$
\lvert x_{n+1} - x_n \rvert \le \tau_x, \qquad \frac{\lvert x_{n+1} - x_n \rvert}{\lvert x_{n+1} \rvert} \le \tau_r, \qquad \lvert f(x_n) \rvert \le \tau_f .
$$

A tiny $\lvert f \rvert$ does not imply a tiny error when $f$ is flat ($\lvert f' \rvert$ small near the root the root is ill-conditioned), and a tiny step does not imply convergence for slowly contracting iterations. Robust codes combine a relative step test with an $f$-residual test, both floored by machine precision.

### Achievable accuracy

Near a simple root, $f(x) \approx f'(x^{*})(x - x^{*})$, and $f$ is computed with absolute noise $\approx \eta$. The root can only be located to within

$$
\lvert x - x^{*} \rvert \lesssim \frac{\eta}{\lvert f'(x^{*}) \rvert},
$$

and for a multiplicity-$m$ root this degrades to $(\eta / \lvert c_m \rvert)^{1/m}$ — a double root computed in double precision yields only about 8 correct digits, no matter the algorithm.

### Cost accounting and hybrid safeguards

Efficiency should be measured per *function evaluation*, not per iteration. The efficiency index is $p^{1/w}$ for order $p$ at $w$ evaluations per step:

| Method | Order $p$ | Evals/step | Index $p^{1/w}$ |
| :--- | :--- | :--- | :--- |
| Bisection | 1 (rate 1/2) | 1 | 1 |
| Newton | 2 | 2 ($f$ and $f'$) | $\sqrt{2} \approx 1.414$ |
| Secant | 1.618 | 1 | $1.618$ |

Hence the classical surprise: *secant beats Newton per evaluation*. **Brent's method** (used by `scipy.optimize.brentq`) keeps a guaranteed bracket, tries inverse quadratic interpolation or secant steps, and accepts them only if they land inside the bracket and shrink it fast enough — otherwise it bisects. It inherits bisection's guarantee and secant-like speed, and is the default choice for 1-D root finding when a bracket is known.

For polynomials, specialized methods (companion-matrix eigenvalues, as in `numpy.roots`) find all roots at once; Newton then polishes.

## 5. Real-World Physics & AI/ML Applications

**Kepler's equation (orbital mechanics).** Converting time to position on an elliptical orbit requires solving the transcendental equation

$$
E - e \sin E = M
$$

for the eccentric anomaly $E$ given mean anomaly $M$ and eccentricity $e$. Since $\frac{d}{dE}(E - e\sin E) = 1 - e\cos E \ge 1 - e \gt 0$, the function is increasing and Newton converges rapidly from $E_0 = M$; every satellite propagator and exoplanet fit solves this equation millions of times.

**Implied volatility (quantitative finance).** The Black–Scholes price $C(\sigma)$ is increasing in volatility $\sigma$; inverting a market price to its implied volatility means solving $C(\sigma) - C_{\mathrm{mkt}} = 0$ — a bracketed, monotone root-finding problem solved with Newton (the derivative, vega, is analytic) or Brent.

**Division and square roots in hardware.** The reciprocal $1/a$ is the root of $f(x) = 1/x - a$; Newton gives the multiplication-only iteration $x_{n+1} = x_n(2 - a x_n)$, which doubles correct bits per step and underlies divider units and the classic fast inverse-square-root trick ($f(x) = 1/x^2 - a$ yields $x_{n+1} = \tfrac{x_n}{2}(3 - a x_n^2)$).

**Machine learning applications.**

- **Logistic regression / GLM fitting**: the 1-D maximum-likelihood condition $\sum_i (y_i - \sigma(\beta x_i)) x_i = 0$ is solved by Newton's method (equivalently, iteratively reweighted least squares); the Hessian is available in closed form.
- **Calibration and temperature scaling**: finding the temperature $T$ that minimizes NLL solves a scalar equation $g'(T) = 0$ by Newton/Brent.
- **Learning-rate and trust-region subproblems**: the Levenberg–Marquardt parameter $\lambda$ solving $\lVert (J^{T}J + \lambda I)^{-1} J^{T} r \rVert = \Delta$ is found by a safeguarded Newton iteration on a 1-D secular equation.
- **Quantile and threshold search**: inverting a CDF ($F(x) - q = 0$) for sampling and conformal prediction uses bracketed Brent iterations.

**Physics equilibria.** Wien's displacement law comes from solving $(x - 5)e^{x} + 5 = 0$ ($x = h\nu/kT$); plasma dispersion relations, beam deflection equations, and shooting methods for boundary-value problems (Topic 08) all reduce to scalar root finding.

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Bisection | $e_n \le (b_0 - a_0)/2^{n+1}$, guaranteed; $n \approx \log_2\frac{b_0-a_0}{\varepsilon}$ |
| Newton | $e_{n+1} \approx \left\lvert \frac{f''(x^{*})}{2f'(x^{*})} \right\rvert e_n^2$ (simple root, local) |
| Secant | $e_{n+1} \approx C e_n e_{n-1}$, order $\varphi \approx 1.618$, best per-eval index |
| Multiple root | Newton linear with factor $\frac{m-1}{m}$; fix with $m f/f'$ |
| Convex + increasing | Newton globally monotone from the right |
| Achievable accuracy | $\lvert x - x^{*} \rvert \lesssim \eta / \lvert f'(x^{*}) \rvert$; $m$-fold roots: $\eta^{1/m}$ |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Bisection, fixed points, Newton, secant | Burden & Faires, *Numerical Analysis* | Ch. 2.1–2.4 |
| Error analysis and convergence orders | Burden & Faires | Ch. 2.4–2.5 |
| Newton–Kantorovich theory, systems | Quarteroni, Sacco & Saleri, *Numerical Mathematics* | Ch. 6–7 |
| Brent's method | Brent, *Algorithms for Minimization without Derivatives* | Ch. 4 |
| Achievable accuracy, conditioning of roots | Heath, *Scientific Computing* | Ch. 5 |
| Convex Newton, global behavior | Ortega & Rheinboldt, *Iterative Solution of Nonlinear Equations* | Ch. 13 |
| Polynomial roots via companion matrices | Trefethen & Bau, *Numerical Linear Algebra* | Lecture 25 |
| Historical context, Newton fractals | Sauer, *Numerical Analysis* | Ch. 1 |

**Primary references.** Burden & Faires (Ch. 2); Heath (Ch. 5); Quarteroni et al. (Chs. 6–7); Brent (1973); Ortega & Rheinboldt (1970); Sauer (Ch. 1).